# 1. 라이브러리 호출

In [ ]:
from pathlib import Path
import ast
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 한글 폰트 설정
# Windows에서는 Malgun Gothic, macOS에서는 AppleGothic, 그 외 환경에서는 NanumGothic을 사용합니다.
import platform

if platform.system() == "Windows":
    plt.rcParams["font.family"] = "Malgun Gothic"
elif platform.system() == "Darwin":
    plt.rcParams["font.family"] = "AppleGothic"
else:
    plt.rcParams["font.family"] = "NanumGothic"

plt.rcParams["axes.unicode_minus"] = False
plt.rcParams["figure.figsize"] = (12, 6)

# pandas 출력 옵션
pd.set_option("display.max_columns", 120)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 180)

# 2. 파일 경로 설정 및 데이터 읽기

In [ ]:
# 원본 데이터 읽기
df_raw01 = pd.read_csv("01_User_Profile.csv")
df_raw02 = pd.read_csv("02_Event_Log.csv")

print("df_raw01 shape :", df_raw01.shape)
print("df_raw02 shape :", df_raw02.shape)

df_raw01 shape : (12500, 6)
df_raw02 shape : (1757262, 5)


In [ ]:
user_profile = df_raw01.copy()
event_log = df_raw02.copy()

In [ ]:
user_profile.head()

,User_ID,가입일자,가입경로,기기,알림수신동의여부,알림수신동의_변경일자
0,U0000001,2025-01-25,오가닉,iOS,True,NaN
1,U0000002,2025-05-06,오가닉,iOS,False,2025-05-24
2,U0000003,2025-05-14,오가닉,iOS,False,NaN
3,U0000004,2025-02-23,퍼포먼스광고,Android,True,NaN
4,U0000005,2025-02-18,퍼포먼스광고,Android,True,NaN


In [ ]:
event_log.head()

,User_ID,Event_Time,Event_Type,Session_ID,알림_유형
0,U0000001,2025-01-25 07:25:45,앱실행,2858201769,NaN
1,U0000001,2025-01-25 07:26:15,온보딩_완료,2858201769,NaN
2,U0000001,2025-01-25 07:26:55,챌린지_탐색,2858201769,NaN
3,U0000001,2025-01-25 07:27:55,챌린지참여,2858201769,NaN
4,U0000001,2025-01-25 20:30:00,알림수신,NaN,광고성


# 3. 데이터 검수

## 3-1 범용 함수

In [ ]:
def check_basic_info(df, df_name, exclude_cols=None):
    """
    행/열 수, 완전 중복 행, 컬럼별 타입/결측/고유값을 한 번에 확인
    """

    print(f"\n{'='*80}")
    print(f"{df_name}의 기본 정보 / 타입 / 결측치 확인")
    print(f"{'='*80}\n")

    df_copied = df.copy()

    # 제외할 컬럼 반영
    if exclude_cols:
        df_copied = df_copied.drop(columns=exclude_cols, errors='ignore')

    # dict, list, set 같은 해시 불가능 값이 들어있는 컬럼은 문자열로 변환
    for col in df_copied.columns:
        try:
            df_copied[col].nunique(dropna=True)
        except TypeError:
            df_copied[col] = df_copied[col].astype(str)

    # 잘못된 데이터프레임 종료
    row_count = len(df_copied)
    if row_count == 0:
        print("데이터프레임이 비어 있습니다.")
        return

    # 전체 요약
    overview_df = pd.DataFrame({
        '항목': ['행 개수', '열 개수', '중복 행 개수'],
        '값': [df_copied.shape[0], df_copied.shape[1], df_copied.duplicated().sum()]
    })

    # 컬럼별 요약
    summary_df = pd.DataFrame({
        '데이터타입': df_copied.dtypes.astype(str),
        '행 개수': df_copied.count(),
        '행 비율(%)': (df_copied.count() / row_count * 100).round(2),
        '결측치 개수': df_copied.isnull().sum(),
        '결측치 비율(%)': (df_copied.isnull().sum() / row_count * 100).round(2),
        '고유값 개수': df_copied.nunique(dropna=True),
        '고유값 비율(%)': (df_copied.nunique(dropna=True) / row_count * 100).round(2)
    }).sort_values(
        by=['결측치 개수', '고유값 개수'],
        ascending=[False, False]
    )

    print("[전체 요약]")
    display(overview_df)

    print("[컬럼별 요약]")
    display(summary_df)

    print("[상위 5행]")
    display(df_copied.head())

In [ ]:
def check_id_duplicates(df, col_name, df_name, top_n=10):
    """
    기준 키로 사용할 컬럼의 중복 여부를 확인한다.
    단일 컬럼 또는 여러 컬럼 조합을 기준으로 확인할 수 있다.
    """

    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 값 중복 확인")
    print(f"{'='*80}")

    # 원본 데이터 훼손 방지
    df_copied = df.copy()

    # 단일 컬럼 기준 확인
    if isinstance(col_name, str):
        if col_name not in df_copied.columns:
            print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
            return

        key_series = df_copied[col_name]
        key_label = col_name

    # 여러 컬럼 조합 기준 확인
    else:
        missing_cols = [col for col in col_name if col not in df_copied.columns]
        if missing_cols:
            print(f"존재하지 않는 컬럼이 있습니다: {missing_cols}")
            return

        key_series = df_copied[col_name].astype(str).agg(" | ".join, axis=1)
        key_label = " + ".join(col_name)

    # 중복 개수 계산
    duplicate_count = key_series.duplicated().sum()

    print("전체 행 수:", len(df_copied))
    print(f"{key_label} 고유 개수:", key_series.nunique(dropna=True))
    print(f"중복 {key_label} 개수:", duplicate_count)

    # 중복 값이 있는 경우 상위 값 확인
    if duplicate_count > 0:
        print("\n[중복 상위 값]")
        dup_summary = key_series.value_counts(dropna=False).reset_index()
        dup_summary.columns = [key_label, "등장 횟수"]
        display(dup_summary[dup_summary["등장 횟수"] > 1].head(top_n))
    else:
        print("중복 값이 없습니다.")
# 이 코드를 진짜 몇번이고 개량하는지 모르겠네

In [ ]:
def check_duplicate_except_cols(df, exclude_cols, df_name, top_n=10):
    """
    특정 컬럼을 제외했을 때, 나머지 값이 완전히 동일한 행이 있는지 확인한다.
    """

    print(f"\n{'='*80}")
    print(f"{df_name}의 특정 컬럼 제외 후 중복 확인")
    print(f"{'='*80}")

    # 원본 데이터 훼손 방지
    df_copied = df.copy()

    # 제외할 컬럼 반영
    check_df = df_copied.drop(columns=exclude_cols, errors='ignore')

    # 잘못된 데이터프레임 종료
    if check_df.shape[1] == 0:
        print("검사할 컬럼이 없습니다.")
        return

    # 특정 컬럼 제외 후 중복 개수 계산
    duplicate_count = check_df.duplicated().sum()

    print("전체 행 수:", len(df_copied))
    print("제외 컬럼:", exclude_cols)
    print("검사 대상 컬럼 수:", check_df.shape[1])
    print("특정 컬럼 제외 후 중복 행 개수:", duplicate_count)

    # 중복 행이 있는 경우 샘플 확인
    if duplicate_count > 0:
        print("\n[중복 행 샘플]")
        dup_mask = check_df.duplicated(keep=False)
        display(df_copied[dup_mask].head(top_n))
    else:
        print("특정 컬럼을 제외해도 중복 행이 없습니다.")
    
# 이 코드를 진짜 몇번이고 개량하는지 모르겠네

In [ ]:
def check_datetime_columns(df, date_cols, df_name):
    """
    날짜 관련 컬럼이 정상적으로 datetime 타입으로 변환되는지 확인한다.
    """

    print(f"\n{'='*80}")
    print(f"{df_name}의 날짜 변환 상태 확인")
    print(f"{'='*80}")

    # 원본 데이터 훼손 방지
    df_copied = df.copy()

    # 날짜 변환 결과 저장
    result_list = []

    for col in date_cols:
        # 컬럼 존재 여부 확인
        if col not in df_copied.columns:
            result_list.append({
                '컬럼명': col,
                '상태': '컬럼 없음',
                '원본 결측치 개수': None,
                '변환 실패 개수': None,
                '최소 날짜': None,
                '최대 날짜': None
            })
            continue

        # 날짜 변환
        converted = pd.to_datetime(df_copied[col], errors='coerce')

        # 원본 결측과 변환 실패 구분
        original_missing = df_copied[col].isnull().sum()
        converted_missing = converted.isnull().sum()
        parse_fail_count = converted_missing - original_missing

        result_list.append({
            '컬럼명': col,
            '상태': '확인 완료',
            '원본 결측치 개수': original_missing,
            '변환 실패 개수': parse_fail_count,
            '최소 날짜': converted.min(),
            '최대 날짜': converted.max()
        })

    # 결과 요약
    result_df = pd.DataFrame(result_list)
    display(result_df)

In [ ]:
def check_statistics_summary(df, df_name, exclude_cols=None):
    """
    수치형 컬럼의 기술통계량과 왜도를 확인한다.
    """

    print(f"\n{'='*80}")
    print(f"{df_name} 변수 기술통계량")
    print(f"{'='*80}\n")

    # 원본 데이터 훼손 방지
    df_copied = df.copy()

    # 제외할 컬럼 반영
    if exclude_cols:
        df_copied = df_copied.drop(columns=exclude_cols, errors='ignore')

    # 수치형 컬럼만 선택
    numeric_df = df_copied.select_dtypes(include='number')

    # 잘못된 데이터프레임 종료
    if numeric_df.shape[1] == 0:
        print("수치형 컬럼이 없습니다.")
        return

    # 기술통계량 계산
    stats_df = numeric_df.describe().T
    stats_df['Skewness'] = numeric_df.skew()

    # 컬럼명 한글로 변경
    stats_df.rename(columns={
        'count': '개수',
        'mean': '평균',
        'std': '표준편차',
        'min': '최솟값',
        '25%': 'Q1의 경계값',
        '50%': '중앙값',
        '75%': 'Q3의 경계값',
        'max': '최댓값',
        'Skewness': '왜도'
    }, inplace=True)

    display(stats_df)

In [ ]:
# 2. IQR 기준 이상치 후보 확인
def check_outliers_iqr(df, df_name, exclude_cols=None):
    """
    수치형 컬럼에 대해 IQR 기준 이상치 후보를 확인한다.
    """

    print(f"\n{'='*80}")
    print(f"{df_name} IQR 기준 이상치 후보 확인")
    print(f"{'='*80}\n")

    # 원본 데이터 훼손 방지
    df_copied = df.copy()

    # 제외할 컬럼 반영
    if exclude_cols:
        df_copied = df_copied.drop(columns=exclude_cols, errors='ignore')

    # 수치형 컬럼만 선택
    numeric_df = df_copied.select_dtypes(include='number')

    # 잘못된 데이터프레임 종료
    if numeric_df.shape[1] == 0:
        print("수치형 컬럼이 없습니다.")
        return

    # IQR 기준 이상치 후보 계산
    result_list = []

    for col in numeric_df.columns:
        q1 = numeric_df[col].quantile(0.25)
        q3 = numeric_df[col].quantile(0.75)
        iqr = q3 - q1

        lower_bound = q1 - 1.5 * iqr
        upper_bound = q3 + 1.5 * iqr

        outlier_count = (
            (numeric_df[col] < lower_bound) |
            (numeric_df[col] > upper_bound)
        ).sum()

        result_list.append({
            '컬럼명': col,
            'Q1': q1,
            'Q3': q3,
            'IQR': iqr,
            '하한값': lower_bound,
            '상한값': upper_bound,
            '이상치 후보 개수': outlier_count,
            '이상치 후보 비율(%)': round(outlier_count / len(numeric_df) * 100, 2)
        })

    # 결과 요약
    result_df = pd.DataFrame(result_list)
    display(result_df)

In [ ]:
def check_category_values(df, df_name, col_name, top_n=10):
    """
    범주형 컬럼에 어떤 값이 있는지 개수와 비율로 확인한다.
    """

    print(f"\n{'='*80}")
    print(f"{df_name}의 {col_name} 값 확인")
    print(f"{'='*80}")

    # 원본 데이터 훼손 방지
    df_copied = df.copy()

    # 컬럼 존재 여부 확인
    if col_name not in df_copied.columns:
        print(f"'{col_name}' 컬럼이 존재하지 않습니다.")
        return

    # 잘못된 데이터프레임 종료
    row_count = len(df_copied)
    if row_count == 0:
        print("데이터프레임이 비어 있습니다.")
        return

    # 범주별 개수 계산, 결측치 포함
    summary_df = df_copied[col_name].value_counts(dropna=False).reset_index()

    # 결과 컬럼명 정리
    summary_df.columns = [col_name, '개수']

    # 범주별 비율 계산
    summary_df['비율(%)'] = (summary_df['개수'] / row_count * 100).round(2)

    print("전체 행 수:", row_count)
    print(f"{col_name} 고유값 개수(결측 포함):", df_copied[col_name].nunique(dropna=False))
    print()

    print("[범주별 요약]")
    display(summary_df.head(top_n))

In [ ]:
def check_user_id_relation(df_profile, df_event):
    """
    User_Profile과 Event_Log의 User_ID 연결 상태를 확인한다.
    """

    print(f"\n{'='*80}")
    print("User_Profile - Event_Log User_ID 연결 확인")
    print(f"{'='*80}")

    # 원본 데이터 훼손 방지
    profile_copied = df_profile.copy()
    event_copied = df_event.copy()

    # 컬럼 존재 여부 확인
    if 'User_ID' not in profile_copied.columns:
        print("User_Profile에 'User_ID' 컬럼이 존재하지 않습니다.")
        return

    if 'User_ID' not in event_copied.columns:
        print("Event_Log에 'User_ID' 컬럼이 존재하지 않습니다.")
        return

    # User_ID 집합 생성
    profile_users = set(profile_copied['User_ID'].dropna())
    event_users = set(event_copied['User_ID'].dropna())

    # 데이터 간 연결 상태 계산
    both_users = profile_users & event_users
    only_profile_users = profile_users - event_users
    only_event_users = event_users - profile_users

    # 연결 상태 요약
    relation_df = pd.DataFrame({
        '항목': [
            '프로필 User_ID 고유 수',
            '이벤트 User_ID 고유 수',
            '양쪽 모두 존재하는 User_ID 수',
            '프로필에만 있는 User_ID 수',
            '이벤트에만 있는 User_ID 수'
        ],
        '값': [
            len(profile_users),
            len(event_users),
            len(both_users),
            len(only_profile_users),
            len(only_event_users)
        ]
    })

    display(relation_df)

    # 프로필에만 존재하는 User_ID 샘플 확인
    if len(only_profile_users) > 0:
        print("\n[프로필에는 있지만 이벤트 로그에는 없는 User_ID 샘플]")
        display(pd.DataFrame({'User_ID': list(only_profile_users)[:10]}))

    # 이벤트 로그에만 존재하는 User_ID 샘플 확인
    if len(only_event_users) > 0:
        print("\n[이벤트 로그에는 있지만 프로필에는 없는 User_ID 샘플]")
        display(pd.DataFrame({'User_ID': list(only_event_users)[:10]}))

## 범용 검수 작업

In [ ]:
# 1. 기본 구조 및 결측값 확인
check_basic_info(user_profile, "User_Profile")
check_basic_info(event_log, "Event_Log")


User_Profile의 기본 정보 / 타입 / 결측치 확인

[전체 요약]


,항목,값
0,행 개수,12500
1,열 개수,6
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수,고유값 비율(%)
알림수신동의_변경일자,object,1976,15.81,10524,84.19,148,1.18
가입경로,object,12363,98.90,137,1.10,2,0.02
기기,object,12379,99.03,121,0.97,2,0.02
알림수신동의여부,object,12384,99.07,116,0.93,2,0.02
User_ID,object,12500,100.00,0,0.00,12500,100.00
가입일자,object,12500,100.00,0,0.00,146,1.17


[상위 5행]


...

,항목,값
0,행 개수,1757262
1,열 개수,5
2,중복 행 개수,0


[컬럼별 요약]


,데이터타입,행 개수,행 비율(%),결측치 개수,결측치 비율(%),고유값 개수,고유값 비율(%)
알림_유형,object,218882,12.46,1538380,87.54,3,0.00
Session_ID,object,1515760,86.26,241502,13.74,736281,41.90
Event_Type,object,1730806,98.49,26456,1.51,10,0.00
Event_Time,object,1757262,100.00,0,0.00,1495136,85.08
User_ID,object,1757262,100.00,0,0.00,12453,0.71


[상위 5행]


,User_ID,Event_Time,Event_Type,Session_ID,알림_유형
0,U0000001,2025-01-25 07:25:45,앱실행,2858201769,NaN
1,U0000001,2025-01-25 07:26:15,온보딩_완료,2858201769,NaN
2,U0000001,2025-01-25 07:26:55,챌린지_탐색,2858201769,NaN
3,U0000001,2025-01-25 07:27:55,챌린지참여,2858201769,NaN
4,U0000001,2025-01-25 20:30:00,알림수신,NaN,광고성


In [ ]:
# 2. 키 중복 확인
check_id_duplicates(user_profile, "User_ID", "User_Profile")
check_id_duplicates(
    event_log,
    ["User_ID", "Event_Time", "Event_Type", "Session_ID", "알림_유형"],
    "Event_Log"
)


User_Profile의 User_ID 값 중복 확인
전체 행 수: 12500
User_ID 고유 개수: 12500
중복 User_ID 개수: 0
중복 값이 없습니다.

Event_Log의 ['User_ID', 'Event_Time', 'Event_Type', 'Session_ID', '알림_유형'] 값 중복 확인
전체 행 수: 1757262
User_ID + Event_Time + Event_Type + Session_ID + 알림_유형 고유 개수: 1757262
중복 User_ID + Event_Time + Event_Type + Session_ID + 알림_유형 개수: 0
중복 값이 없습니다.


In [ ]:
# 3. ID 제외 후 값 중복 확인

# User_ID만 제외하고 나머지 값이 모두 같은 경우
check_duplicate_except_cols(
    user_profile,
    ["User_ID"],
    "User_Profile"
)

# User_ID, Session_ID를 제외하고 나머지 로그 값이 같은 경우
check_duplicate_except_cols(
    event_log,
    ["User_ID", "Session_ID"],
    "Event_Log"
)


User_Profile의 특정 컬럼 제외 후 중복 확인
전체 행 수: 12500
제외 컬럼: ['User_ID']
검사 대상 컬럼 수: 5
특정 컬럼 제외 후 중복 행 개수: 9121

[중복 행 샘플]


,User_ID,가입일자,가입경로,기기,알림수신동의여부,알림수신동의_변경일자
0,U0000001,2025-01-25,오가닉,iOS,True,NaN
2,U0000003,2025-05-14,오가닉,iOS,False,NaN
3,U0000004,2025-02-23,퍼포먼스광고,Android,True,NaN
4,U0000005,2025-02-18,퍼포먼스광고,Android,True,NaN
5,U0000006,2025-02-07,퍼포먼스광고,Android,True,NaN
7,U0000008,2025-02-27,오가닉,iOS,False,NaN
8,U0000009,2025-05-20,퍼포먼스광고,iOS,False,NaN
9,U0000010,2025-04-21,오가닉,iOS,False,NaN
10,U0000011,2025-02-24,오가닉,iOS,True,NaN
11,U0000012,2025-04-21,오가닉,iOS,True,NaN



Event_Log의 특정 컬럼 제외 후 중복 확인
전체 행 수: 1757262
제외 컬럼: ['User_ID', 'Session_ID']
검사 대상 컬럼 수: 3
특정 컬럼 제외 후 중복 행 개수: 87794

[중복 행 샘플]


,User_ID,Event_Time,Event_Type,Session_ID,알림_유형
9,U0000001,2025-01-26 15:18:00,알림수신,NaN,리마인드
13,U0000001,2025-01-26 20:11:12,앱실행,0504a5087b,NaN
15,U0000001,2025-01-26 22:58:09,앱실행,06566985d2,NaN
48,U0000001,2025-01-29 18:23:00,알림수신,NaN,리마인드
111,U0000001,2025-02-04 19:43:00,알림수신,NaN,챌린지_알림
112,U0000001,2025-02-05 09:47:42,앱실행,dbddce03dc,NaN
113,U0000001,2025-02-05 16:14:00,알림수신,NaN,리마인드
124,U0000001,2025-02-06 22:12:29,앱실행,d0e98dba23,NaN
151,U0000001,2025-02-12 09:20:00,알림수신,NaN,리마인드
159,U0000001,2025-02-13 10:37:00,알림수신,NaN,리마인드


In [ ]:
# 4. 날짜 컬럼 확인
check_datetime_columns(
    user_profile,
    ["가입일자", "알림수신동의_변경일자"],
    "User_Profile"
)
check_datetime_columns(
    event_log,
    ["Event_Time"],
    "Event_Log"
)


User_Profile의 날짜 변환 상태 확인


,컬럼명,상태,원본 결측치 개수,변환 실패 개수,최소 날짜,최대 날짜
0,가입일자,확인 완료,0,0,2025-01-01,2025-05-26
1,알림수신동의_변경일자,확인 완료,10524,0,2025-01-21,2025-06-30



Event_Log의 날짜 변환 상태 확인


,컬럼명,상태,원본 결측치 개수,변환 실패 개수,최소 날짜,최대 날짜
0,Event_Time,확인 완료,0,0,2025-01-01 07:00:07,2025-06-30 22:59:51


In [ ]:
# 5. 범주형 값 분포 확인
print("user_profile")
check_category_values(user_profile, "User_Profile", "가입경로") 
check_category_values(user_profile, "User_Profile", "기기") 
check_category_values(user_profile, "User_Profile", "알림수신동의여부")

print("\n event_log")

check_category_values(event_log, "Event_Log", "Event_Type") 
check_category_values(event_log, "Event_Log", "알림_유형")

user_profile

User_Profile의 가입경로 값 확인
전체 행 수: 12500
가입경로 고유값 개수(결측 포함): 3

[범주별 요약]


,가입경로,개수,비율(%)
0,퍼포먼스광고,6852,54.82
1,오가닉,5511,44.09
2,NaN,137,1.10



User_Profile의 기기 값 확인
전체 행 수: 12500
기기 고유값 개수(결측 포함): 3

[범주별 요약]


,기기,개수,비율(%)
0,iOS,7175,57.40
1,Android,5204,41.63
2,NaN,121,0.97



User_Profile의 알림수신동의여부 값 확인
전체 행 수: 12500
알림수신동의여부 고유값 개수(결측 포함): 3

[범주별 요약]


,알림수신동의여부,개수,비율(%)
0,True,7984,63.87
1,False,4400,35.20
2,NaN,116,0.93



 event_log

Event_Log의 Event_Type 값 확인
전체 행 수: 1757262
Event_Type 고유값 개수(결측 포함): 11

[범주별 요약]


,Event_Type,개수,비율(%)
0,앱실행,728657,41.47
1,수면기록,242978,13.83
2,알림수신,194324,11.06
3,운동기록,131269,7.47
4,마음챙김,130344,7.42
5,식단기록,101366,5.77
6,챌린지참여,96829,5.51
7,챌린지_탐색,78101,4.44
8,NaN,26456,1.51
9,알림오픈,21219,1.21



Event_Log의 알림_유형 값 확인
전체 행 수: 1757262
알림_유형 고유값 개수(결측 포함): 4

[범주별 요약]


,알림_유형,개수,비율(%)
0,NaN,1538380,87.54
1,리마인드,85830,4.88
2,광고성,78262,4.45
3,챌린지_알림,54790,3.12


In [ ]:
# 6. 데이터 간 키 연결 상태 확인
check_user_id_relation( user_profile, event_log)


User_Profile - Event_Log User_ID 연결 확인


,항목,값
0,프로필 User_ID 고유 수,12500
1,이벤트 User_ID 고유 수,12453
2,양쪽 모두 존재하는 User_ID 수,12453
3,프로필에만 있는 User_ID 수,47
4,이벤트에만 있는 User_ID 수,0



[프로필에는 있지만 이벤트 로그에는 없는 User_ID 샘플]


,User_ID
0,U0011781
1,U0009092
2,U0002163
3,U0011193
4,U0003574
5,U0001646
6,U0008711
7,U0002389
8,U0001915
9,U0000537
